# Raw 20 ms SBP PCA: T12 chronological session holdout

This notebook measures the dimensionality of **instantaneous 20 ms × 128-channel SBP activity**. A PCA row is one native neural bin. It does not use flattened decoder windows, GRU hidden states, phoneme labels or timings, CCA, or trajectory alignment.

## Provenance and interpretation

- **Reported:** Spalding et al., *Shared latent representations of speech production for cross-patient speech decoding* (Nature Communications, 2026), Methods “Latent dynamics extraction,” fit patient-specific PCA over the neural channel dimension and retained the number of PCs explaining 90% of input variance.
- **Adapted:** this notebook uses intracortical clipped-FP16 SBP from T12, native 20 ms bins, a chronological future-session holdout, and two normalization regimes. It is a Utah-array dimensionality diagnostic, not a reproduction of the paper.
- **Repository observation:** the canonical Brain-to-Text 2024 speech view is the first 128 area-6v SBP channels. No decoder-derived information enters this analysis.
- **AI assistance:** Codex generated the notebook structure and analytical code from the approved design. The synthetic equivalence test, cache audit, contract assertions, and artifact reopen checks are included for human review.

Run the smoke configuration first. Then set `SMOKE_MODE = False` for the complete analysis.


In [ ]:
# Mount Drive, clone/update the repository, and install only missing packages.

from pathlib import Path
import importlib.util
import os
import subprocess
import sys

try:
    import google.colab  # type: ignore  # noqa: F401
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

if IN_COLAB:
    from google.colab import drive
    drive.mount('/content/drive')

REPO_URL = 'https://github.com/ethan-read/utah-ssl.git'
REPO_DIR = Path('/content/utah-ssl') if IN_COLAB else Path.cwd()

if IN_COLAB:
    if not REPO_DIR.exists():
        subprocess.run(['git', 'clone', REPO_URL, str(REPO_DIR)], check=True)
    else:
        status = subprocess.run(
            ['git', '-C', str(REPO_DIR), 'status', '--porcelain'],
            check=True, capture_output=True, text=True,
        )
        if status.stdout.strip():
            print('Using existing checkout with local changes; automatic pull skipped.')
        else:
            subprocess.run(
                ['git', '-C', str(REPO_DIR), 'pull', '--ff-only', 'origin', 'main'],
                check=True,
            )

os.chdir(REPO_DIR)
if str(REPO_DIR) not in sys.path:
    sys.path.insert(0, str(REPO_DIR))

required_packages = {
    'numpy': 'numpy',
    'pandas': 'pandas',
    'matplotlib': 'matplotlib',
}
missing = [pip_name for module, pip_name in required_packages.items()
           if importlib.util.find_spec(module) is None]
if missing:
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', *missing], check=True)

print({'in_colab': IN_COLAB, 'repo_dir': str(REPO_DIR), 'installed': missing})


In [ ]:
# Experiment configuration. Change SMOKE_MODE only after the smoke run succeeds.

import json
import tempfile
from collections import defaultdict
from datetime import datetime, timezone

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

from utah_ssl.cache_identity import compute_dataset_cache_source_signature
from utah_ssl.canonical_data import (
    CanonicalShardAccessor,
    load_canonical_manifest,
    load_canonical_metadata,
    validate_canonical_dataset,
)
from utah_ssl.experiment_contract import DatasetPlan, SignalSpec

SMOKE_MODE = True
SMOKE_MAX_TRIALS_PER_SESSION = 4
OVERWRITE_OUTPUT = False

DATASET = 'brain2text24'
SOURCE_SPLIT = 'competition_train'
SUBJECT_ID = 't12'
BIN_SIZE_MS = 20
N_CHANNELS = 128
EPSILON = 1e-8
TOP_K_REFERENCE = 6
EXPECTED_SBP_CLIP_THRESHOLD = 12500.0

EXPECTED_CACHE_VARIANT = 'cache_v1_sbpclip12500_fp16_raw'
EXPECTED_HOLDOUT_SESSION_IDS = (
    't12.2022.08.13',
    't12.2022.08.18',
    't12.2022.08.23',
    't12.2022.08.25',
)

DRIVE_ROOT = (
    Path('/content/drive/MyDrive')
    if IN_COLAB
    else Path('/Users/home/My Drive')
)
UTAH_SSL_ROOT = DRIVE_ROOT / 'utah_ssl'
CACHE_ROOT = UTAH_SSL_ROOT / 'data' / EXPECTED_CACHE_VARIANT
OUTPUT_ROOT = UTAH_SSL_ROOT / 'outputs' / 'neural_trajectories'
BASE_RUN_NAME = 'raw_20ms_sbp_pca_t12_chronological_v1'
RUN_NAME = BASE_RUN_NAME + ('_smoke' if SMOKE_MODE else '')
OUTPUT_DIR = OUTPUT_ROOT / RUN_NAME
if OUTPUT_DIR.exists() and not OVERWRITE_OUTPUT:
    raise FileExistsError(
        f'Refusing to start because output already exists: {OUTPUT_DIR}. '
        'Set OVERWRITE_OUTPUT=True only after reviewing it.'
    )
if OUTPUT_DIR.exists():
    print('Existing output will be moved to a timestamped backup after analysis:', OUTPUT_DIR)
WORK_DIR = Path(tempfile.mkdtemp(prefix=f'{RUN_NAME}_'))

DATASET_PLAN = DatasetPlan.from_mapping({DATASET: (SOURCE_SPLIT,)})
SIGNAL_SPEC = SignalSpec.sbp_only(
    sbp_dim=N_CHANNELS,
    column_start=0,
    missing_channel_policy='error',
)

print({
    'smoke_mode': SMOKE_MODE,
    'cache_root': str(CACHE_ROOT),
    'output_dir': str(OUTPUT_DIR),
    'work_dir': str(WORK_DIR),
    'dataset_plan': DATASET_PLAN.to_dict(),
    'signal_spec': SIGNAL_SPEC.to_dict(),
})


## Contract and cache preflight

The notebook positively selects only T12 `competition_train` sentence trials. It verifies every selected shard is physically FP16 with at least 128 SBP channels. The maintained cache-audit CLI checks representative arrays; the full analysis scans every selected value for finiteness while accumulating statistics.


In [ ]:
# Resolve the canonical manifest, enforce the chronological split, and audit the cache.

import hashlib

dataset_root, manifest_path, metadata_path = validate_canonical_dataset(
    CACHE_ROOT, dataset=DATASET,
)
if CACHE_ROOT.name != EXPECTED_CACHE_VARIANT:
    raise ValueError(
        f'Expected clipped raw SBP cache {EXPECTED_CACHE_VARIANT!r}, got {CACHE_ROOT.name!r}'
    )

metadata = load_canonical_metadata(metadata_path)
manifest_rows = load_canonical_manifest(manifest_path)
with manifest_path.open() as handle:
    manifest_payloads = [json.loads(line) for line in handle if line.strip()]
payload_by_id = {str(row['example_id']): row for row in manifest_payloads}

feature_layout = metadata.get('feature_layout') or {}
if int(metadata.get('bin_size_ms', -1)) != BIN_SIZE_MS:
    raise ValueError(f'Expected metadata bin_size_ms={BIN_SIZE_MS}: {metadata.get("bin_size_ms")}')
if int(feature_layout.get('n_sbp_features', -1)) != N_CHANNELS:
    raise ValueError(f'Expected exactly {N_CHANNELS} SBP channels: {feature_layout}')
if metadata.get('sbp_storage_dtype') != 'float16':
    raise ValueError(
        f'Expected metadata sbp_storage_dtype=float16, got {metadata.get("sbp_storage_dtype")!r}'
    )
if float(metadata.get('sbp_clip_threshold', -1)) != EXPECTED_SBP_CLIP_THRESHOLD:
    raise ValueError(
        f'Expected SBP clip threshold {EXPECTED_SBP_CLIP_THRESHOLD}, '
        f'got {metadata.get("sbp_clip_threshold")!r}'
    )

candidate_rows = [
    row for row in manifest_rows
    if row.source_split == SOURCE_SPLIT and row.subject_id == SUBJECT_ID
]
if len(candidate_rows) != 8800:
    raise ValueError(f'Expected 8,800 T12 competition_train rows, found {len(candidate_rows):,}')

for row in candidate_rows:
    payload = payload_by_id[row.example_id]
    if int(payload.get('bin_size_ms', -1)) != BIN_SIZE_MS:
        raise ValueError(f'Non-20 ms row: {row.example_id}')
    if int(payload.get('source_bin_size_ms', -1)) != BIN_SIZE_MS:
        raise ValueError(
            f'Expected native source_bin_size_ms={BIN_SIZE_MS}: {row.example_id}'
        )
    if bool(payload.get('resampled_to_20ms', False)):
        raise ValueError(f'Expected native 20 ms bins, found resampled row: {row.example_id}')
    if str(payload.get('task_name')) != 'sentences':
        raise ValueError(f'Unexpected task in competition_train: {payload.get("task_name")}')
    if not SIGNAL_SPEC.row_is_compatible(
        has_tx=row.n_tx_features > 0,
        has_sbp=row.n_sbp_features > 0,
        n_tx_features=row.n_tx_features,
        n_sbp_features=row.n_sbp_features,
    ):
        raise ValueError(f'Row violates SignalSpec: {row.example_id}')

session_ids = tuple(sorted({row.session_id for row in candidate_rows}))
if len(session_ids) != 24:
    raise ValueError(f'Expected 24 competition_train sessions, found {len(session_ids)}')
if session_ids[-4:] != EXPECTED_HOLDOUT_SESSION_IDS:
    raise ValueError(
        f'Chronological holdout changed. Expected {EXPECTED_HOLDOUT_SESSION_IDS}, '
        f'found {session_ids[-4:]}'
    )
train_session_ids = session_ids[:-4]
heldout_session_ids = session_ids[-4:]
if train_session_ids[-1] != 't12.2022.08.11':
    raise ValueError(f'Unexpected final fit session: {train_session_ids[-1]}')
if set(train_session_ids) & set(heldout_session_ids):
    raise AssertionError('Fit and held-out sessions overlap')

physical_shards = sorted({row.shard_relpath for row in candidate_rows})
for shard_relpath in physical_shards:
    sbp_path = CACHE_ROOT / shard_relpath / 'sbp.npy'
    sbp = np.load(sbp_path, mmap_mode='r')
    if sbp.dtype != np.float16:
        raise ValueError(f'Expected physical clipped-FP16 SBP at {sbp_path}, got {sbp.dtype}')
    if sbp.ndim != 2 or sbp.shape[1] != N_CHANNELS:
        raise ValueError(f'Invalid SBP shape at {sbp_path}: {sbp.shape}')
    del sbp

rows_by_session = defaultdict(list)
for row in candidate_rows:
    rows_by_session[row.session_id].append(row)
for session_id in rows_by_session:
    rows_by_session[session_id].sort(key=lambda row: row.example_id)

selected_rows_by_session = {
    session_id: (
        rows[:SMOKE_MAX_TRIALS_PER_SESSION] if SMOKE_MODE else rows
    )
    for session_id, rows in sorted(rows_by_session.items())
}
selected_rows = [
    row
    for session_id in session_ids
    for row in selected_rows_by_session[session_id]
]
if {row.session_id for row in selected_rows} != set(session_ids):
    raise AssertionError('Smoke/full selection must retain every chronological session')

cache_audit_path = WORK_DIR / 'cache_audit.json'
audit_command = [
    sys.executable,
    str(REPO_DIR / 'utah_ssl' / 'scripts' / 'audit_cache_roots.py'),
    '--cache-root', str(CACHE_ROOT),
    '--dataset', DATASET,
    '--segment-bins', '1',
    '--feature-mode', 'sbp_only',
    '--sample-shards', '3',
    '--output-json', str(cache_audit_path),
]
subprocess.run(audit_command, cwd=str(REPO_DIR), check=True)
cache_audit = json.loads(cache_audit_path.read_text())
root_audits = cache_audit.get('root_audits') or []
if len(root_audits) != 1:
    raise RuntimeError(f'Expected one cache root audit, found {len(root_audits)}')
dataset_audit = (root_audits[0].get('datasets') or {}).get(DATASET)
if not isinstance(dataset_audit, dict):
    raise RuntimeError(f'Cache audit omitted dataset {DATASET!r}')
array_totals = ((dataset_audit.get('array_check') or {}).get('totals') or {})
required_array_failures = {
    key: int(array_totals.get(key, 0) or 0)
    for key in (
        'missing_sbp_files',
        'missing_time_offsets_files',
        'sbp_width_mismatches',
        'row_count_mismatches',
        'manifest_length_mismatches',
        'bad_example_index',
    )
}
required_array_failures = {key: value for key, value in required_array_failures.items() if value}
if required_array_failures:
    raise RuntimeError(f'Cache audit found required-array failures: {required_array_failures}')
audit_findings = list(dataset_audit.get('findings') or [])
if 'missing_array_files' in audit_findings and int(array_totals.get('missing_tx_files', 0) or 0) > 0:
    # TX is outside this explicit SBP-only SignalSpec; only required SBP/offset failures block above.
    audit_findings.remove('missing_array_files')
if audit_findings:
    raise RuntimeError(f'Cache audit reported blocking findings: {audit_findings}')
feature_audit = (dataset_audit.get('feature_mode_audits') or {}).get('sbp_only') or {}
if not bool(feature_audit.get('prepare_cache_context_ok')):
    raise RuntimeError(
        'SBP-only runtime cache validation failed: '
        f'{feature_audit.get("prepare_cache_context_error")}'
    )

git_commit = subprocess.run(
    ['git', 'rev-parse', 'HEAD'], cwd=str(REPO_DIR),
    check=True, capture_output=True, text=True,
).stdout.strip()
git_status = subprocess.run(
    ['git', 'status', '--porcelain'], cwd=str(REPO_DIR),
    check=True, capture_output=True, text=True,
).stdout.splitlines()

source_signature = compute_dataset_cache_source_signature({DATASET: CACHE_ROOT})
provenance = {
    'source': {
        'title': 'Shared latent representations of speech production for cross-patient speech decoding',
        'authors': 'Spalding et al.',
        'venue_year': 'Nature Communications, 2026',
        'doi': '10.1038/s41467-026-75455-1',
        'evidence_location': 'Methods: Latent dynamics extraction, PDF page 20',
        'reported_element': 'PCA over neural channels; retain PCs explaining 90% of input variance',
    },
    'adaptation': {
        'local_implementation': 'experiments/manifolds/notebooks/raw_20ms_sbp_pca.ipynb',
        'classification': 'Adapted',
        'deviation': (
            'T12 intracortical SBP at native 20 ms cadence; chronological session holdout; '
            'session-wise and training-global z-score conditions; no CCA or timing labels'
        ),
        'validation': 'Synthetic SVD equivalence, cache audit, contract assertions, artifact reopen checks',
    },
    'repository_observation': {
        'dataset_plan': DATASET_PLAN.to_dict(),
        'signal_spec': SIGNAL_SPEC.to_dict(),
        'cache_source_signature': source_signature,
        'manifest_sha256': hashlib.sha256(manifest_path.read_bytes()).hexdigest(),
        'metadata_sha256': hashlib.sha256(metadata_path.read_bytes()).hexdigest(),
    },
    'upstream_code_reused': False,
    'license_note': 'No paper or released-code implementation was copied into this notebook.',
    'ai_assistance': (
        'Codex generated the notebook structure and analytical code from the approved design; '
        'human review and empirical interpretation remain required.'
    ),
}
(WORK_DIR / 'provenance.json').write_text(json.dumps(provenance, indent=2) + '\n')
config = {
    'run_name': RUN_NAME,
    'created_utc': datetime.now(timezone.utc).isoformat(),
    'smoke_mode': SMOKE_MODE,
    'smoke_max_trials_per_session': SMOKE_MAX_TRIALS_PER_SESSION if SMOKE_MODE else None,
    'dataset_plan': DATASET_PLAN.to_dict(),
    'signal_spec': SIGNAL_SPEC.to_dict(),
    'cache_root': str(CACHE_ROOT),
    'cache_variant': CACHE_ROOT.name,
    'cache_source_signature': source_signature,
    'dataset': DATASET,
    'source_split': SOURCE_SPLIT,
    'subject_id': SUBJECT_ID,
    'bin_size_ms': BIN_SIZE_MS,
    'smoothing': 'none',
    'augmentation': 'none',
    'fit_session_ids': list(train_session_ids),
    'heldout_session_ids': list(heldout_session_ids),
    'normalization_conditions': {
        'session_zscore': {
            'role': 'primary',
            'fit': 'each session uses its own unlabeled mean/std',
            'heldout_classification': 'transductive nuisance removal',
        },
        'train_global_zscore': {
            'role': 'control',
            'fit': 'mean/std from fit sessions only',
            'heldout_classification': 'inductive future-session control',
        },
    },
    'pca_observation': 'one native 20 ms x 128-channel SBP vector',
    'bin_weighting': 'equal weight per 20 ms bin',
    'top_k_reference': TOP_K_REFERENCE,
    'git_commit': git_commit,
    'git_worktree_dirty': bool(git_status),
    'ai_assistance': (
        'Codex generated the notebook structure and analytical code from the approved design; '
        'human review and empirical interpretation remain required.'
    ),
}
(WORK_DIR / 'config.json').write_text(json.dumps(config, indent=2) + '\n')

preflight_summary = {
    'candidate_rows': len(candidate_rows),
    'selected_rows': len(selected_rows),
    'candidate_bins': int(sum(row.n_time_bins or 0 for row in candidate_rows)),
    'selected_bins': int(sum(row.n_time_bins or 0 for row in selected_rows)),
    'sessions': len(session_ids),
    'fit_sessions': len(train_session_ids),
    'heldout_sessions': len(heldout_session_ids),
    'physical_shards': len(physical_shards),
    'source_signature': source_signature,
}
print(json.dumps(preflight_summary, indent=2))


## Exact PCA from sufficient statistics

For each session the notebook stores only (n), (sum x), (sum x^2), and (X^T X). These are enough to derive both normalization conditions, exact 128-dimensional PCA, and held-out reconstruction without retaining the full bin matrix.


In [ ]:
# Numerical helpers for sufficient statistics, PCA, and held-out reconstruction.

def empty_sufficient_stats(dim):
    return {
        'n': 0,
        'sum': np.zeros(dim, dtype=np.float64),
        'sum_sq': np.zeros(dim, dtype=np.float64),
        'cross': np.zeros((dim, dim), dtype=np.float64),
    }


def update_sufficient_stats(stats, values):
    x = np.asarray(values, dtype=np.float64)
    if x.ndim != 2 or x.shape[1] != stats['sum'].shape[0]:
        raise ValueError(f'Unexpected feature shape: {x.shape}')
    if not np.isfinite(x).all():
        locations = np.argwhere(~np.isfinite(x))
        raise ValueError(f'Nonfinite neural values found; first locations: {locations[:5].tolist()}')
    stats['n'] += int(x.shape[0])
    stats['sum'] += x.sum(axis=0)
    stats['sum_sq'] += np.einsum('ij,ij->j', x, x, optimize=True)
    stats['cross'] += x.T @ x


def combine_sufficient_stats(items):
    items = list(items)
    if not items:
        raise ValueError('Cannot combine an empty statistics collection')
    combined = empty_sufficient_stats(items[0]['sum'].shape[0])
    for item in items:
        combined['n'] += int(item['n'])
        combined['sum'] += item['sum']
        combined['sum_sq'] += item['sum_sq']
        combined['cross'] += item['cross']
    return combined


def mean_and_std(stats, epsilon=EPSILON):
    if stats['n'] < 2:
        raise ValueError(f'Need at least two bins, found {stats["n"]}')
    mean = stats['sum'] / stats['n']
    variance = stats['sum_sq'] / stats['n'] - np.square(mean)
    variance = np.maximum(variance, 0.0)
    std = np.sqrt(variance)
    safe_std = np.maximum(std, epsilon)
    return mean, safe_std


def scatter_about_center(stats, center):
    center = np.asarray(center, dtype=np.float64)
    return (
        stats['cross']
        - np.outer(center, stats['sum'])
        - np.outer(stats['sum'], center)
        + stats['n'] * np.outer(center, center)
    )


def scale_scatter(scatter, std):
    std = np.asarray(std, dtype=np.float64)
    return scatter / np.outer(std, std)


def session_zscore_scatter(stats):
    mean, std = mean_and_std(stats)
    return scale_scatter(scatter_about_center(stats, mean), std)


def pca_from_scatter(scatter, n_observations):
    covariance = np.asarray(scatter, dtype=np.float64) / max(int(n_observations) - 1, 1)
    covariance = 0.5 * (covariance + covariance.T)
    eigenvalues, eigenvectors = np.linalg.eigh(covariance)
    order = np.argsort(eigenvalues)[::-1]
    eigenvalues = eigenvalues[order]
    eigenvectors = eigenvectors[:, order]

    tolerance = max(1.0, float(np.max(np.abs(eigenvalues)))) * 1e-10
    if float(eigenvalues.min()) < -tolerance:
        raise AssertionError(f'Covariance has materially negative eigenvalue {eigenvalues.min()}')
    eigenvalues = np.maximum(eigenvalues, 0.0)

    # Resolve arbitrary eigenvector signs for stable saved loadings and figures.
    for column in range(eigenvectors.shape[1]):
        pivot = int(np.argmax(np.abs(eigenvectors[:, column])))
        if eigenvectors[pivot, column] < 0:
            eigenvectors[:, column] *= -1

    total = float(eigenvalues.sum())
    if not np.isfinite(total) or total <= 0:
        raise ValueError(f'PCA variance must be positive and finite, got {total}')
    ratio = eigenvalues / total
    cumulative = np.cumsum(ratio)
    if np.any(np.diff(cumulative) < -1e-12):
        raise AssertionError('Cumulative explained variance is not monotonic')
    if not np.isclose(cumulative[-1], 1.0, atol=1e-9):
        raise AssertionError(f'Final cumulative explained variance is {cumulative[-1]}')
    return {
        'eigenvalues': eigenvalues,
        'components': eigenvectors,
        'explained_variance_ratio': ratio,
        'cumulative_explained_variance': cumulative,
    }


def component_count_for_threshold(cumulative, threshold):
    return int(np.searchsorted(cumulative, float(threshold), side='left') + 1)


def participation_ratio(eigenvalues):
    values = np.asarray(eigenvalues, dtype=np.float64)
    return float(np.square(values.sum()) / np.square(values).sum())


def reconstruction_curve(scatter, components, n_observations):
    scatter = 0.5 * (np.asarray(scatter) + np.asarray(scatter).T)
    projected_energy = np.sum(components * (scatter @ components), axis=0)
    tolerance = max(1.0, float(np.trace(scatter))) * 1e-10
    if float(projected_energy.min()) < -tolerance:
        raise AssertionError(f'Negative projected energy: {projected_energy.min()}')
    projected_energy = np.maximum(projected_energy, 0.0)
    cumulative_energy = np.cumsum(projected_energy)
    total_energy = float(np.trace(scatter))
    if total_energy <= 0 or not np.isfinite(total_energy):
        raise ValueError(f'Held-out energy must be positive and finite, got {total_energy}')
    captured_fraction = np.clip(cumulative_energy / total_energy, 0.0, 1.0)
    residual_energy = np.maximum(total_energy - cumulative_energy, 0.0)
    mse = residual_energy / (int(n_observations) * components.shape[0])
    if np.any(np.diff(captured_fraction) < -1e-12):
        raise AssertionError('Held-out reconstruction is not monotonic')
    if np.any(np.diff(mse) > 1e-12):
        raise AssertionError('Held-out reconstruction MSE increased with component count')
    return {
        'captured_fraction': captured_fraction,
        'mse': mse,
        'total_energy': total_energy,
    }


In [ ]:
# Synthetic equivalence test: sufficient-statistics PCA must match direct SVD.

rng = np.random.default_rng(7)
latent = rng.normal(size=(600, 4))
mixing = rng.normal(size=(4, 12))
synthetic_train = latent @ mixing + 0.15 * rng.normal(size=(600, 12))
synthetic_test = rng.normal(size=(175, 4)) @ mixing + 0.15 * rng.normal(size=(175, 12))

synthetic_stats = empty_sufficient_stats(synthetic_train.shape[1])
update_sufficient_stats(synthetic_stats, synthetic_train)
synthetic_mean, _ = mean_and_std(synthetic_stats)
synthetic_scatter = scatter_about_center(synthetic_stats, synthetic_mean)
synthetic_pca = pca_from_scatter(synthetic_scatter, synthetic_stats['n'])

_, singular_values, vt = np.linalg.svd(
    synthetic_train - synthetic_train.mean(axis=0),
    full_matrices=False,
)
direct_eigenvalues = np.square(singular_values) / (len(synthetic_train) - 1)
np.testing.assert_allclose(
    synthetic_pca['eigenvalues'], direct_eigenvalues, rtol=1e-10, atol=1e-10,
)

test_stats = empty_sufficient_stats(synthetic_test.shape[1])
update_sufficient_stats(test_stats, synthetic_test)
test_scatter = scatter_about_center(test_stats, synthetic_mean)
sufficient_curve = reconstruction_curve(
    test_scatter, synthetic_pca['components'], test_stats['n'],
)['captured_fraction']

direct_scores = (synthetic_test - synthetic_mean) @ vt.T
direct_curve = np.cumsum(np.square(direct_scores).sum(axis=0))
direct_curve /= np.square(synthetic_test - synthetic_mean).sum()
np.testing.assert_allclose(sufficient_curve, direct_curve, rtol=1e-10, atol=1e-10)

print('Synthetic PCA and held-out reconstruction checks passed.')


## Stream selected raw SBP bins

Full mode reads all 8,800 `competition_train` trials. Smoke mode reads the configured prefix from every session so the split and all validation paths remain exercised.


In [ ]:
# Accumulate exact per-session sufficient statistics in one pass.

session_stats = {
    session_id: empty_sufficient_stats(N_CHANNELS)
    for session_id in session_ids
}
accessor = CanonicalShardAccessor(CACHE_ROOT)

processed_rows = 0
try:
    for session_id in session_ids:
        rows = selected_rows_by_session[session_id]
        for row in rows:
            values = accessor.load_features(row, signal_spec=SIGNAL_SPEC)
            if values.shape[0] != int(row.n_time_bins or values.shape[0]):
                raise ValueError(
                    f'Manifest/array bin mismatch for {row.example_id}: '
                    f'{row.n_time_bins} vs {values.shape[0]}'
                )
            update_sufficient_stats(session_stats[session_id], values)
            processed_rows += 1
            if processed_rows % 500 == 0:
                print(f'Processed {processed_rows:,}/{len(selected_rows):,} trials')
finally:
    accessor.close()

for session_id, stats in session_stats.items():
    expected_bins = sum(int(row.n_time_bins or 0) for row in selected_rows_by_session[session_id])
    if stats['n'] != expected_bins:
        raise AssertionError(f'{session_id}: accumulated {stats["n"]}, expected {expected_bins}')

stats_session_ids = np.asarray(session_ids)
stats_n = np.asarray([session_stats[s]['n'] for s in session_ids], dtype=np.int64)
stats_sum = np.stack([session_stats[s]['sum'] for s in session_ids])
stats_sum_sq = np.stack([session_stats[s]['sum_sq'] for s in session_ids])
stats_cross = np.stack([session_stats[s]['cross'] for s in session_ids])
np.savez_compressed(
    WORK_DIR / 'sufficient_statistics.npz',
    session_ids=stats_session_ids,
    n=stats_n,
    sum=stats_sum,
    sum_sq=stats_sum_sq,
    cross=stats_cross,
)

session_inventory = pd.DataFrame([
    {
        'session_id': session_id,
        'partition': 'fit' if session_id in train_session_ids else 'heldout',
        'trials': len(selected_rows_by_session[session_id]),
        'bins': session_stats[session_id]['n'],
    }
    for session_id in session_ids
])
session_inventory.to_csv(WORK_DIR / 'session_inventory.csv', index=False)
display(session_inventory)


In [ ]:
# Fit both PCA conditions and evaluate future-session reconstruction.

fit_stats = combine_sufficient_stats(session_stats[s] for s in train_session_ids)
fit_global_mean, fit_global_std = mean_and_std(fit_stats)

condition_inputs = {}

session_train_scatter = sum(
    (session_zscore_scatter(session_stats[s]) for s in train_session_ids),
    start=np.zeros((N_CHANNELS, N_CHANNELS), dtype=np.float64),
)
condition_inputs['session_zscore'] = {
    'train_scatter': session_train_scatter,
    'heldout_scatters': {
        s: session_zscore_scatter(session_stats[s])
        for s in heldout_session_ids
    },
}

global_train_scatter = scale_scatter(
    scatter_about_center(fit_stats, fit_global_mean),
    fit_global_std,
)
condition_inputs['train_global_zscore'] = {
    'train_scatter': global_train_scatter,
    'heldout_scatters': {
        s: scale_scatter(
            scatter_about_center(session_stats[s], fit_global_mean),
            fit_global_std,
        )
        for s in heldout_session_ids
    },
}

if set(train_session_ids) & set(heldout_session_ids):
    raise AssertionError('Held-out sessions leaked into the fit partition')
if tuple(config['fit_session_ids']) != train_session_ids:
    raise AssertionError('Serialized fit-session contract does not match the analysis')

analysis_results = {}
variance_rows = []
loading_rows = []
curve_rows = []
metric_rows = []
summary = {
    'run_name': RUN_NAME,
    'smoke_mode': SMOKE_MODE,
    'fit_session_ids': list(train_session_ids),
    'heldout_session_ids': list(heldout_session_ids),
    'conditions': {},
}

for condition, inputs in condition_inputs.items():
    pca = pca_from_scatter(inputs['train_scatter'], fit_stats['n'])
    cumulative = pca['cumulative_explained_variance']
    k50 = component_count_for_threshold(cumulative, 0.50)
    k80 = component_count_for_threshold(cumulative, 0.80)
    k90 = component_count_for_threshold(cumulative, 0.90)
    pr = participation_ratio(pca['eigenvalues'])

    for pc_index in range(N_CHANNELS):
        variance_rows.append({
            'condition': condition,
            'pc': pc_index + 1,
            'eigenvalue': pca['eigenvalues'][pc_index],
            'explained_variance_ratio': pca['explained_variance_ratio'][pc_index],
            'cumulative_explained_variance': cumulative[pc_index],
        })
    for pc_index in range(TOP_K_REFERENCE):
        for channel in range(N_CHANNELS):
            loading_rows.append({
                'condition': condition,
                'pc': pc_index + 1,
                'channel': channel,
                'loading': pca['components'][channel, pc_index],
            })

    heldout_curves = {}
    pooled_scatter = np.zeros((N_CHANNELS, N_CHANNELS), dtype=np.float64)
    pooled_n = 0
    for session_id in heldout_session_ids:
        scatter = inputs['heldout_scatters'][session_id]
        n_bins = session_stats[session_id]['n']
        heldout_curves[session_id] = reconstruction_curve(
            scatter, pca['components'], n_bins,
        )
        pooled_scatter += scatter
        pooled_n += n_bins
    heldout_curves['pooled'] = reconstruction_curve(
        pooled_scatter, pca['components'], pooled_n,
    )

    for evaluation, curve in heldout_curves.items():
        n_bins = pooled_n if evaluation == 'pooled' else session_stats[evaluation]['n']
        for k_index in range(N_CHANNELS):
            curve_rows.append({
                'condition': condition,
                'evaluation': evaluation,
                'k': k_index + 1,
                'n_bins': n_bins,
                'captured_fraction': curve['captured_fraction'][k_index],
                'reconstruction_mse': curve['mse'][k_index],
            })
        for label, k in (('top6', TOP_K_REFERENCE), ('train_k90', k90)):
            metric_rows.append({
                'condition': condition,
                'evaluation': evaluation,
                'metric_point': label,
                'k': k,
                'n_bins': n_bins,
                'captured_fraction': curve['captured_fraction'][k - 1],
                'reconstruction_mse': curve['mse'][k - 1],
            })

    summary['conditions'][condition] = {
        'top6_cumulative_train_variance': float(cumulative[TOP_K_REFERENCE - 1]),
        'k50': k50,
        'k80': k80,
        'k90': k90,
        'participation_ratio': pr,
        'pooled_heldout_top6_captured_fraction': float(
            heldout_curves['pooled']['captured_fraction'][TOP_K_REFERENCE - 1]
        ),
        'pooled_heldout_k90_captured_fraction': float(
            heldout_curves['pooled']['captured_fraction'][k90 - 1]
        ),
        'pooled_heldout_top6_reconstruction_mse': float(
            heldout_curves['pooled']['mse'][TOP_K_REFERENCE - 1]
        ),
        'pooled_heldout_k90_reconstruction_mse': float(
            heldout_curves['pooled']['mse'][k90 - 1]
        ),
    }
    analysis_results[condition] = {
        'pca': pca,
        'heldout_curves': heldout_curves,
    }

variance_df = pd.DataFrame(variance_rows)
loadings_df = pd.DataFrame(loading_rows)
curves_df = pd.DataFrame(curve_rows)
metrics_df = pd.DataFrame(metric_rows)

variance_df.to_csv(WORK_DIR / 'explained_variance.csv', index=False)
loadings_df.to_csv(WORK_DIR / 'top6_channel_loadings.csv', index=False)
curves_df.to_csv(WORK_DIR / 'heldout_reconstruction_curves.csv', index=False)
metrics_df.to_csv(WORK_DIR / 'heldout_reconstruction_metrics.csv', index=False)
(WORK_DIR / 'summary.json').write_text(json.dumps(summary, indent=2) + '\n')

display(pd.DataFrame(summary['conditions']).T)
display(metrics_df[metrics_df['evaluation'] == 'pooled'])


In [ ]:
# Generate the fixed result figures.

condition_labels = {
    'session_zscore': 'Session-wise z-score (transductive)',
    'train_global_zscore': 'Training-global z-score (inductive)',
}
colors = {
    'session_zscore': '#3366cc',
    'train_global_zscore': '#dc3912',
}

fig, axes = plt.subplots(1, 2, figsize=(13, 4.5), constrained_layout=True)
for condition, result in analysis_results.items():
    ratio = result['pca']['explained_variance_ratio']
    cumulative = result['pca']['cumulative_explained_variance']
    x = np.arange(1, N_CHANNELS + 1)
    axes[0].plot(x, ratio, label=condition_labels[condition], color=colors[condition])
    axes[1].plot(x, cumulative, label=condition_labels[condition], color=colors[condition])
axes[0].set_yscale('log')
axes[0].set_xlabel('Principal component')
axes[0].set_ylabel('Individual explained-variance ratio')
axes[0].set_title('Training eigenspectrum')
axes[1].axhline(0.90, color='black', linestyle='--', linewidth=1, label='90% threshold')
axes[1].axvline(TOP_K_REFERENCE, color='gray', linestyle=':', linewidth=1, label='Top 6')
axes[1].set_xlabel('Number of principal components')
axes[1].set_ylabel('Cumulative explained-variance ratio')
axes[1].set_ylim(0, 1.01)
axes[1].set_title('Training cumulative variance')
for ax in axes:
    ax.grid(alpha=0.2)
    ax.legend(fontsize=8)
fig.savefig(WORK_DIR / 'explained_variance_curves.png', dpi=180)
plt.show()

fig, axes = plt.subplots(1, 2, figsize=(14, 4.8), constrained_layout=True, sharey=True)
for ax, condition in zip(axes, condition_inputs):
    curves = analysis_results[condition]['heldout_curves']
    x = np.arange(1, N_CHANNELS + 1)
    for session_id in heldout_session_ids:
        ax.plot(x, curves[session_id]['captured_fraction'], alpha=0.55, linewidth=1,
                label=session_id.replace('t12.', ''))
    ax.plot(x, curves['pooled']['captured_fraction'], color='black', linewidth=2.2,
            label='pooled holdout')
    k90 = summary['conditions'][condition]['k90']
    ax.axvline(TOP_K_REFERENCE, color='gray', linestyle=':', linewidth=1)
    ax.axvline(k90, color='black', linestyle='--', linewidth=1)
    ax.set_title(condition_labels[condition])
    ax.set_xlabel('Training PCs used for reconstruction')
    ax.grid(alpha=0.2)
    ax.legend(fontsize=7)
axes[0].set_ylabel('Held-out reconstruction: captured energy fraction')
axes[0].set_ylim(0, 1.01)
fig.savefig(WORK_DIR / 'heldout_reconstruction_curves.png', dpi=180)
plt.show()

fig, axes = plt.subplots(2, 1, figsize=(15, 5.5), constrained_layout=True)
for ax, condition in zip(axes, condition_inputs):
    loadings = analysis_results[condition]['pca']['components'][:, :TOP_K_REFERENCE].T
    vmax = float(np.max(np.abs(loadings)))
    image = ax.imshow(
        loadings, aspect='auto', cmap='coolwarm', vmin=-vmax, vmax=vmax,
        interpolation='nearest',
    )
    ax.set_title(condition_labels[condition])
    ax.set_ylabel('PC')
    ax.set_yticks(np.arange(TOP_K_REFERENCE), np.arange(1, TOP_K_REFERENCE + 1))
    ax.set_xlabel('Area-6v channel index')
    fig.colorbar(image, ax=ax, shrink=0.8, label='Loading')
fig.savefig(WORK_DIR / 'top6_channel_loadings.png', dpi=180)
plt.show()


## Persist, reopen, and verify artifacts

Results are staged locally and then copied into a versioned Drive directory. An existing result is never silently replaced. If `OVERWRITE_OUTPUT=True`, the old directory is renamed to a timestamped backup before promotion.


In [ ]:
# Promote staged artifacts to Drive and verify they can be reopened.

import shutil
import uuid

required_artifacts = (
    'config.json',
    'provenance.json',
    'cache_audit.json',
    'sufficient_statistics.npz',
    'session_inventory.csv',
    'explained_variance.csv',
    'top6_channel_loadings.csv',
    'heldout_reconstruction_curves.csv',
    'heldout_reconstruction_metrics.csv',
    'summary.json',
    'explained_variance_curves.png',
    'heldout_reconstruction_curves.png',
    'top6_channel_loadings.png',
)
missing_staged = [name for name in required_artifacts if not (WORK_DIR / name).is_file()]
if missing_staged:
    raise FileNotFoundError(f'Missing staged artifacts: {missing_staged}')

OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)
if OUTPUT_DIR.exists():
    if not OVERWRITE_OUTPUT:
        raise FileExistsError(
            f'Refusing to overwrite existing output: {OUTPUT_DIR}. '
            'Set OVERWRITE_OUTPUT=True only after reviewing it.'
        )
    backup_suffix = datetime.now(timezone.utc).strftime('%Y%m%dT%H%M%SZ')
    backup_dir = OUTPUT_DIR.with_name(f'{OUTPUT_DIR.name}_backup_{backup_suffix}')
    OUTPUT_DIR.rename(backup_dir)
    print('Moved previous output to recoverable backup:', backup_dir)

staging_dir = OUTPUT_ROOT / f'.{RUN_NAME}.staging.{uuid.uuid4().hex}'
shutil.copytree(WORK_DIR, staging_dir)
staging_dir.rename(OUTPUT_DIR)

missing_drive = [name for name in required_artifacts if not (OUTPUT_DIR / name).is_file()]
if missing_drive:
    raise FileNotFoundError(f'Drive promotion incomplete: {missing_drive}')

reopened_summary = json.loads((OUTPUT_DIR / 'summary.json').read_text())
reopened_config = json.loads((OUTPUT_DIR / 'config.json').read_text())
reopened_metrics = pd.read_csv(OUTPUT_DIR / 'heldout_reconstruction_metrics.csv')
with np.load(OUTPUT_DIR / 'sufficient_statistics.npz') as reopened_stats:
    reopened_session_ids = tuple(reopened_stats['session_ids'].tolist())
    reopened_counts = reopened_stats['n'].copy()

if reopened_summary['run_name'] != RUN_NAME:
    raise AssertionError('Reopened summary has the wrong run name')
if tuple(reopened_config['heldout_session_ids']) != heldout_session_ids:
    raise AssertionError('Reopened configuration has the wrong holdout')
if reopened_session_ids != session_ids:
    raise AssertionError('Reopened sufficient statistics have the wrong sessions')
if not np.array_equal(reopened_counts, stats_n):
    raise AssertionError('Reopened sufficient-statistics counts changed')
if len(reopened_metrics) != 2 * 5 * 2:
    raise AssertionError(f'Unexpected held-out metric row count: {len(reopened_metrics)}')

print('Artifact reopen checks passed.')
print('Saved:', OUTPUT_DIR)
print(json.dumps(reopened_summary['conditions'], indent=2))


## Reading the result

The primary question is whether the training top-six cumulative variance is materially larger than the approximately 30% previously observed for flattened 280 ms input windows. Also inspect (k_{90}), participation ratio, and whether the training basis reconstructs future sessions.

The two conditions answer different questions:

- **Session-wise z-score:** how low-dimensional is within-session activity after each day’s offsets and gains are removed? Its held-out result is transductive because it uses unlabeled held-out-session statistics.
- **Training-global z-score:** how well does a basis and normalization learned on earlier dates carry into later dates with day drift intact?

These are dimensionality and stability diagnostics only. They do not establish phoneme-specific trajectories or justify PCA-CCA by themselves.


In [ ]:
# Final Colab teardown. Run only after the artifact reopen checks pass.

if IN_COLAB:
    from google.colab import drive, runtime
    drive.flush_and_unmount()
    runtime.unassign()
else:
    print('Local run: no Colab Drive mount or runtime to release.')
